In [1]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_excel(r"D:\НДВ\Коммерция с Нашего дома рф\Характеристики\Характеристики.xlsx")


In [15]:
df["Год готовности"] = pd.to_datetime(
    df["Дата готовности"],
    errors="coerce"
).dt.year

In [11]:
df["Год готовности"].unique()

array([2024, 2025, 2021, 2026, 2023, 2022], dtype=int32)

In [5]:
df2 = pd.read_excel(r"D:\НДВ\Коммерция с Нашего дома рф\Лоты\Лоты.xlsx")


In [16]:
# Оставляем только нужные помещения
df2_filtered = df2[
    (df2["Type"] == "Нежилое помещение для коммерческого использования")
    & (df2["Floornumber"] == 1)
]

# Считаем сумму площади и количество помещений по каждому ЖК
stats = (
    df2_filtered
    .groupby("Project_id", as_index=False)
    .agg(
        Commercial_area=("Totalarea", "sum"),
        Commercial_count=("Project_id", "size")
    )
)

# Добавляем результаты в датафрейм с характеристиками
df = df.merge(stats, on="Project_id", how="left")

# Если для некоторых ЖК помещений нет, заполняем нулями
df[["Commercial_area", "Commercial_count"]] = (
    df[["Commercial_area", "Commercial_count"]]
    .fillna(0)
)

# При необходимости количество сделать целым числом
df["Commercial_count"] = df["Commercial_count"].astype(int)

In [17]:
df2_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10862 entries, 467 to 296774
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   City         10862 non-null  object 
 1   Project_id   10862 non-null  int64  
 2   Type         10862 non-null  object 
 3   Floornumber  10862 non-null  int64  
 4   Price        0 non-null      float64
 5   Status       0 non-null      object 
 6   Totalarea    10862 non-null  float64
dtypes: float64(2), int64(2), object(3)
memory usage: 678.9+ KB


In [18]:
df.to_excel(r"D:\НДВ\Коммерция с Нашего дома рф\Характеристики\Характеристики2.xlsx", index=False)